# 深大微堆低浓铀模型材料定义

本Notebook用于逐步建立深大微堆（MNSR-SZ）低浓铀统一参考情景所需的OpenMC材料库，并最终生成`build/materials.xml`。每一种材料都在独立单元格中定义，便于逐项理解、检查和修改。本Notebook不依赖`materials.py`，也不包含几何、settings、tallies或优化算法。

In [1]:
import sys
from numbers import Real
from pathlib import Path
from typing import cast
import openmc
from openmc.data import atomic_mass

TEMPERATURE_K = cast(Real, 293.15)

print("Python解释器:", sys.executable)
print("OpenMC版本:", openmc.__version__)
print("OpenMC安装位置:", openmc.__file__)

Python解释器: ~/miniconda3/bin/python
OpenMC版本: 0.15.3.dev146+gc0f302db6
OpenMC安装位置: ~/.local/lib/python3.13/site-packages/openmc/__init__.py


## 1 核数据库路径设置

`CROSS_SECTIONS`必须指向实际存在的`cross_sections.xml`文件本身，不能只填写它所在的文件夹。当前填写的是本机已经确认存在的路径；如果数据目录发生变化，只需要修改下面这一处。后续材料定义不会再次设置核数据库路径，也不读取`.env`或隐藏环境变量。

In [2]:
import os

CROSS_SECTIONS = Path(
    os.environ.get("OPENMC_CROSS_SECTIONS", "~/openmc_data/endfb-vii.1-hdf5/cross_sections.xml")
).expanduser()

if not CROSS_SECTIONS.is_file():
    raise FileNotFoundError(
        "找不到OpenMC核数据库配置文件，请修改CROSS_SECTIONS。\n"
        f"当前填写路径：{CROSS_SECTIONS}"
    )

openmc.config["cross_sections"] = str(CROSS_SECTIONS)

print("使用的核数据库：", CROSS_SECTIONS)

使用的核数据库： ~/openmc_data/endfb-vii.1-hdf5/cross_sections.xml


下面的辅助单元只用于寻找用户主目录下可能存在的`cross_sections.xml`。它不会自动选择任何结果，以免误用不匹配的数据库。找到正确文件后，应将其路径复制到上一个单元的`CROSS_SECTIONS`中。

In [3]:
search_root = Path.home()
candidates = list(search_root.rglob("cross_sections.xml"))

print("找到的cross_sections.xml候选文件：")
for index, path in enumerate(candidates):
    print(index, path)

找到的cross_sections.xml候选文件：
0 ~/openmc_data/endfb-vii.1-hdf5/cross_sections.xml


## 2 材料组成辅助函数

下面只定义两个简短辅助函数。`normalize_weight_fractions`把合金配方中的质量份额归一化；`weight_to_atomic_fractions`利用OpenMC提供的同位素原子质量，将同位素质量比例转换为材料定义所需的相对原子比例。材料的主体定义仍会在后续单元逐项展示。

In [4]:
def normalize_weight_fractions(
    composition: dict[str, float],
) -> dict[str, float]:
    """检查并归一化元素质量份额。"""
    if any(fraction < 0.0 for fraction in composition.values()):
        raise ValueError("所有质量份额必须大于或等于0。")

    total = sum(composition.values())
    if total <= 0.0:
        raise ValueError("质量份额总和必须大于0。")

    return {
        element: fraction / total
        for element, fraction in composition.items()
    }


def weight_to_atomic_fractions(
    isotope_weight_fractions: dict[str, float],
) -> dict[str, float]:
    """使用同位素原子质量将质量份额转换并归一化为原子份额。"""
    normalized_weights = normalize_weight_fractions(
        isotope_weight_fractions
    )
    unnormalized_atoms = {
        isotope: weight_fraction / atomic_mass(isotope)
        for isotope, weight_fraction in normalized_weights.items()
    }
    total_atoms = sum(unnormalized_atoms.values())

    return {
        isotope: atom_amount / total_atoms
        for isotope, atom_amount in unnormalized_atoms.items()
    }

## 3 低浓铀UO₂燃料

12.5 wt%表示U-235占全部铀元素质量的12.5%，并不表示它占整个UO₂材料质量的12.5%。由于富集度超过5%，这里不使用OpenMC的自动`enrichment`经验展开，而是按照IAEA MNSR低浓铀方案显式输入U-234、U-235、U-236和U-238的铀内质量组成。转换后的铀同位素原子份额共同构成1份铀，再加入2份氧，因此U:O原子数比保持1:2。统一参考情景采用10.6 g/cm³的燃料密度和293.15 K温度。

In [5]:
import warnings

uranium_weight_fractions = {
    "U234": 0.0020,
    "U235": 0.1250,
    "U236": 0.0025,
    "U238": 0.8705,
}

uranium_atom_fractions = weight_to_atomic_fractions(
    uranium_weight_fractions
)

with warnings.catch_warnings(record=True) as fuel_definition_warnings:
    warnings.simplefilter("always")

    leu_fuel = openmc.Material(name="LEU UO2 12.5wt%")

    for nuclide, fraction in uranium_atom_fractions.items():
        leu_fuel.add_nuclide(nuclide, fraction)

    leu_fuel.add_element("O", 2.0)
    leu_fuel.set_density("g/cm3", 10.6)
    leu_fuel.temperature = TEMPERATURE_K

print("铀同位素原子份额：")
for nuclide, fraction in uranium_atom_fractions.items():
    print(f"{nuclide}: {fraction:.12f}")

铀同位素原子份额：
U234: 0.002030905779
U235: 0.126389969504
U236: 0.002517072902
U238: 0.869062051815


下面单独检查燃料定义：铀内原子份额必须归一化为1，1份铀对应2份氧，并反算确认U-235占铀质量约12.5%。同时检查定义过程中没有出现超过5%富集度的自动展开警告。

In [6]:
import math

uranium_atom_total = sum(uranium_atom_fractions.values())
oxygen_atom_amount = 2.0

uranium_mass_total = sum(
    fraction * atomic_mass(nuclide)
    for nuclide, fraction in uranium_atom_fractions.items()
)
reconstructed_u235_weight_fraction = (
    uranium_atom_fractions["U235"]
    * atomic_mass("U235")
    / uranium_mass_total
)

warning_messages = [
    str(captured.message)
    for captured in fuel_definition_warnings
]

assert math.isclose(uranium_atom_total, 1.0, abs_tol=1.0e-12)
assert math.isclose(oxygen_atom_amount / uranium_atom_total, 2.0)
assert math.isclose(
    reconstructed_u235_weight_fraction,
    0.125,
    abs_tol=1.0e-12,
)
assert not any(
    "enrichment" in message.lower() and "5%" in message
    for message in warning_messages
)

print("铀原子份额之和：", uranium_atom_total)
print("U:O原子数比：1:", oxygen_atom_amount)
print(
    "反算U-235铀内质量百分比：",
    reconstructed_u235_weight_fraction * 100.0,
)
print("燃料定义阶段捕获的警告：", warning_messages)

铀原子份额之和： 1.0
U:O原子数比：1: 2.0
反算U-235铀内质量百分比： 12.499999999999998
燃料定义阶段捕获的警告： []


## 4 Zircaloy-4燃料包壳

Zircaloy-4是由锆、锡、铁和铬组成的锆合金，不能使用100%的Zr-90替代。以下配方是元素质量份额，先归一化，再以`percent_type="wo"`加入天然元素。该材料用于燃料包壳，密度为6.5 g/cm³，温度为293.15 K。

In [7]:
zircaloy4_composition = normalize_weight_fractions({
    "Zr": 97.93,
    "Sn": 1.70,
    "Fe": 0.24,
    "Cr": 0.13,
})

zircaloy4 = openmc.Material(name="Zircaloy-4")
for element, fraction in zircaloy4_composition.items():
    zircaloy4.add_element(
        element,
        fraction,
        percent_type="wo",
    )

zircaloy4.set_density("g/cm3", 6.5)
zircaloy4.temperature = TEMPERATURE_K

print("Zircaloy-4归一化质量份额：", zircaloy4_composition)

Zircaloy-4归一化质量份额： {'Zr': 0.9793000000000001, 'Sn': 0.017, 'Fe': 0.0024, 'Cr': 0.0013}


## 5 Al-303-1假元件与连接杆材料

Al-303-1用于5根铝合金假元件以及第8圈4根固定连接杆。原始配方可能因小数取舍而不严格等于100，因此先自动归一化，再按质量份额添加天然元素。统一模型采用2.7 g/cm³密度和293.15 K温度。

In [8]:
al_303_1_composition = normalize_weight_fractions({
    "Al": 97.75,
    "Fe": 1.20,
    "Si": 1.00,
    "Cu": 0.005,
    "Zn": 0.003,
    "Mn": 0.003,
    "Mg": 0.005,
    "Ti": 0.005,
    "B": 0.0001,
})

al_303_1 = openmc.Material(name="Al-303-1")
for element, fraction in al_303_1_composition.items():
    al_303_1.add_element(element, fraction, percent_type="wo")

al_303_1.set_density("g/cm3", 2.7)
al_303_1.temperature = TEMPERATURE_K

print("Al-303-1归一化质量份额：", al_303_1_composition)

Al-303-1归一化质量份额： {'Al': 0.9777825791653787, 'Fe': 0.012003469002541733, 'Si': 0.010002890835451445, 'Cu': 5.0014454177257224e-05, 'Zn': 3.0008672506354334e-05, 'Mn': 3.0008672506354334e-05, 'Mg': 5.0014454177257224e-05, 'Ti': 5.0014454177257224e-05, 'B': 1.0002890835451446e-06}


## 6 LT-21反应堆容器与控制棒导管材料

LT-21用于反应堆容器、中央控制棒导管以及统一简化模型中的其他铝合金结构。各元素按质量份额归一化添加；模型密度为2.7 g/cm³，温度为293.15 K。

In [9]:
al_lt21_composition = normalize_weight_fractions({
    "Al": 98.1648,
    "Si": 0.9,
    "Mg": 0.675,
    "Fe": 0.2,
    "Ni": 0.03,
    "Cu": 0.01,
    "Mn": 0.01,
    "Ti": 0.01,
    "Cd": 0.0001,
    "B": 0.0001,
})

al_lt21 = openmc.Material(name="LT-21")
for element, fraction in al_lt21_composition.items():
    al_lt21.add_element(element, fraction, percent_type="wo")

al_lt21.set_density("g/cm3", 2.7)
al_lt21.temperature = TEMPERATURE_K

print("LT-21归一化质量份额：", al_lt21_composition)

LT-21归一化质量份额： {'Al': 0.981648, 'Si': 0.009000000000000001, 'Mg': 0.006750000000000001, 'Fe': 0.002, 'Ni': 0.0003, 'Cu': 0.0001, 'Mn': 0.0001, 'Ti': 0.0001, 'Cd': 1e-06, 'B': 1e-06}


## 7 SS-304控制棒包壳材料

SS-304不锈钢用于中央镉控制棒的外包管。以下元素配方按质量份额归一化添加，采用7.8 g/cm³密度和293.15 K温度。

In [10]:
ss304_composition = normalize_weight_fractions({
    "Fe": 70.845,
    "Cr": 18.0,
    "Ni": 8.0,
    "Mn": 2.0,
    "Si": 1.0,
    "C": 0.08,
    "P": 0.045,
    "S": 0.03,
})

ss304 = openmc.Material(name="SS-304")
for element, fraction in ss304_composition.items():
    ss304.add_element(element, fraction, percent_type="wo")

ss304.set_density("g/cm3", 7.8)
ss304.temperature = TEMPERATURE_K

print("SS-304归一化质量份额：", ss304_composition)

SS-304归一化质量份额： {'Fe': 0.70845, 'Cr': 0.18, 'Ni': 0.08, 'Mn': 0.02, 'Si': 0.01, 'C': 0.0008, 'P': 0.00045, 'S': 0.0003}


## 8 轻水慢化剂与冷却剂

轻水中的H与O原子数比为2:1，它在模型中同时作为中子慢化剂和冷却剂。`c_H_in_H2O`热散射表用于描述水分子中束缚氢在热中子能区的散射行为。

In [11]:
water = openmc.Material(name="Light Water")
water.add_nuclide("H1", 2.0)
water.add_element("O", 1.0)
water.set_density("g/cm3", 0.99825)
water.temperature = TEMPERATURE_K
water.add_s_alpha_beta("c_H_in_H2O")

print("轻水材料定义完成：", water.name)

轻水材料定义完成： Light Water


## 9 金属铍反射层

这里采用忽略少量杂质的简化天然金属铍模型，用于统一计算情景中的侧铍反射层。`c_Be_in_Be`用于描述金属铍晶格中的热中子散射。

In [12]:
beryllium = openmc.Material(name="Beryllium Reflector")
beryllium.add_element("Be", 1.0)
beryllium.set_density("g/cm3", 1.85)
beryllium.temperature = TEMPERATURE_K
beryllium.add_s_alpha_beta("c_Be_in_Be")

print("金属铍反射层材料定义完成：", beryllium.name)

金属铍反射层材料定义完成： Beryllium Reflector


## 10 中央控制棒镉吸收体

控制棒吸收体使用天然镉同位素组成，而不是100%的Cd-113。该材料只会在几何模型的控制棒插入状态下使用，密度为8.65 g/cm³，温度为293.15 K。

In [13]:
cadmium = openmc.Material(name="Natural Cadmium")
cadmium.add_element("Cd", 1.0)
cadmium.set_density("g/cm3", 8.65)
cadmium.temperature = TEMPERATURE_K

print("天然镉吸收体材料定义完成：", cadmium.name)

天然镉吸收体材料定义完成： Natural Cadmium


## 11 材料库汇总

现在把8个唯一材料对象放入`openmc.Materials`。`material_map`按照几何用途建立别名，使同一个材料可以服务于多个结构，例如Al-303-1同时用于假元件、连接杆和控制棒中心铝棒。下面打印材料ID、名称、密度、温度和主要用途，便于核对。

In [14]:
materials = openmc.Materials([
    leu_fuel,
    zircaloy4,
    al_303_1,
    al_lt21,
    ss304,
    water,
    beryllium,
    cadmium,
])

material_map = {
    "fuel": leu_fuel,
    "fuel_clad": zircaloy4,
    "dummy": al_303_1,
    "tie_rod": al_303_1,
    "control_rod_clad": ss304,
    "control_rod_inner": al_303_1,
    "guide_tube": al_lt21,
    "vessel": al_lt21,
    "water": water,
    "beryllium": beryllium,
    "cadmium": cadmium,
}

material_usages = {
    "LEU UO2 12.5wt%": "低浓铀燃料芯体",
    "Zircaloy-4": "燃料包壳",
    "Al-303-1": "假元件、连接杆、控制棒中心铝棒",
    "LT-21": "反应堆容器、控制棒导管",
    "SS-304": "控制棒外包管",
    "Light Water": "慢化剂、冷却剂和结构间隙水",
    "Beryllium Reflector": "侧铍反射层",
    "Natural Cadmium": "插入状态下的控制棒吸收体",
}

for material in materials:
    print(
        f"ID={material.id:<2} "
        f"名称={material.name:<24} "
        f"密度={material.density} {material.density_units:<6} "
        f"温度={material.temperature} K "
        f"用途={material_usages[material.name]}"
    )

ID=1  名称=LEU UO2 12.5wt%          密度=10.6 g/cm3  温度=293.15 K 用途=低浓铀燃料芯体
ID=2  名称=Zircaloy-4               密度=6.5 g/cm3  温度=293.15 K 用途=燃料包壳
ID=3  名称=Al-303-1                 密度=2.7 g/cm3  温度=293.15 K 用途=假元件、连接杆、控制棒中心铝棒
ID=4  名称=LT-21                    密度=2.7 g/cm3  温度=293.15 K 用途=反应堆容器、控制棒导管
ID=5  名称=SS-304                   密度=7.8 g/cm3  温度=293.15 K 用途=控制棒外包管
ID=6  名称=Light Water              密度=0.99825 g/cm3  温度=293.15 K 用途=慢化剂、冷却剂和结构间隙水
ID=7  名称=Beryllium Reflector      密度=1.85 g/cm3  温度=293.15 K 用途=侧铍反射层
ID=8  名称=Natural Cadmium          密度=8.65 g/cm3  温度=293.15 K 用途=插入状态下的控制棒吸收体


## 12 材料定义检查

导出前进行基础一致性检查：材料数量和用途键必须完整；燃料密度必须正确；Zircaloy-4和天然镉不能退化为单一同位素；水与铍必须带有所需热散射表；所有材料名称必须唯一。检查通过后才继续导出。

In [15]:
assert len(materials) == 8

assert set([
    "fuel",
    "fuel_clad",
    "dummy",
    "tie_rod",
    "control_rod_clad",
    "control_rod_inner",
    "guide_tube",
    "vessel",
    "water",
    "beryllium",
    "cadmium",
]).issubset(material_map)

fuel_density = leu_fuel.density
assert fuel_density is not None
assert math.isclose(fuel_density, 10.6)

zircaloy_nuclides = {
    nuclide.name for nuclide in zircaloy4.nuclides
}
cadmium_nuclides = {
    nuclide.name for nuclide in cadmium.nuclides
}

assert zircaloy_nuclides != {"Zr90"}
assert cadmium_nuclides != {"Cd113"}

water_xml = water.to_xml_element()
beryllium_xml = beryllium.to_xml_element()
assert any(
    table.get("name") == "c_H_in_H2O"
    for table in water_xml.findall("sab")
)
assert any(
    table.get("name") == "c_Be_in_Be"
    for table in beryllium_xml.findall("sab")
)

material_names = [material.name for material in materials]
assert len(material_names) == len(set(material_names))

print("材料库检查通过。")

材料库检查通过。


## 13 导出OpenMC材料文件

检查通过后，将材料库导出到项目的`build/materials.xml`。这样不会覆盖项目根目录中已有的`materials.xml`。该XML文件随后可由上层`openmc.Model`与几何、settings和tallies共同使用。

In [16]:
BUILD_DIR = Path("build")
BUILD_DIR.mkdir(exist_ok=True)

materials.export_to_xml(
    path=BUILD_DIR / "materials.xml"
)

exported_xml = (BUILD_DIR / "materials.xml").resolve()
print("导出文件：", exported_xml)
print("文件存在：", exported_xml.is_file())
print("文件大小：", exported_xml.stat().st_size, "bytes")

导出文件： ~/SZU_MNSR_Opt/build/materials.xml
文件存在： True
文件大小： 7473 bytes


## 14 查看XML内容

下面的可选单元读取导出XML的前30行，帮助理解OpenMC如何保存材料、核素、密度和热散射表。XML应由Notebook重新生成，不需要手工编辑。

In [17]:
xml_path = BUILD_DIR / "materials.xml"

with xml_path.open(
    "r",
    encoding="utf-8",
) as file:
    for _ in range(30):
        line = file.readline()
        if not line:
            break
        print(line.rstrip())

print()
print("材料库检查通过。")

<?xml version='1.0' encoding='utf-8'?>
<materials>
  <material id="1" name="LEU UO2 12.5wt%" depletable="true" temperature="293.15">
    <density value="10.6" units="g/cm3"/>
    <nuclide name="U234" ao="0.0020309057791265276"/>
    <nuclide name="U235" ao="0.12638996950386247"/>
    <nuclide name="U236" ao="0.0025170729019679786"/>
    <nuclide name="U238" ao="0.869062051815043"/>
    <nuclide name="O16" ao="1.999242"/>
    <nuclide name="O17" ao="0.000758"/>
  </material>
  <material id="2" name="Zircaloy-4" temperature="293.15">
    <density value="6.5" units="g/cm3"/>
    <nuclide name="Zr90" wo="0.49656500876792037"/>
    <nuclide name="Zr91" wo="0.10949443082821599"/>
    <nuclide name="Zr92" wo="0.16920444763331058"/>
    <nuclide name="Zr94" wo="0.17520758482456905"/>
    <nuclide name="Zr96" wo="0.028828527945984138"/>
    <nuclide name="Sn112" wo="0.0001554467865622746"/>
    <nuclide name="Sn114" wo="0.000107656303149813"/>
    <nuclide name="Sn115" wo="5.594648295678121e-05